# OceanEmbed v2.1 — cyclone-aware, uncertainty-aware subsurface reconstruction

Surface satellite data (OSTIA SST, SMOS/SMAP SSS, DUACS SSH, OSCAR currents, CCMP winds; today + 2 previous days) →
**ViT + FNO satellite-embedding encoder** → **U-Net decoder** → temperature at 35 depths (0–902 m) **with calibrated uncertainty**.
Trained on GLORYS with **cyclone-aware physics losses** (D26, TCHP, D20, MLD + stability + steric/SSH),
validated on independent **ARGO** floats, plus a **real-event case study** (deep depression, 1 Aug 2023).

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.


## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU - switch runtime to T4 GPU"
# Drive is OPTIONAL (only used to back up results). If mounting fails, we continue without it.
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OK = True
except Exception as e:
    print("Drive not mounted (", e, ") -> continuing; results stay in /content, download them at the end.")


In [ ]:
%cd /content
!rm -rf Ocean_Embed
!git clone -q -b pinn-pipeline https://github.com/dhriti-goswami/Ocean_Embed.git
%cd /content/Ocean_Embed
!pip install -q netCDF4
!git log --oneline -3

## 2. Data + paths
Downloads the team cube from the shared Drive link. Results are written locally and backed up to your Drive after each stage.


In [ ]:
import os, shutil
# Data: team cube shared on Google Drive (link from Kaashvi)
DATA_FILE_ID = "1lV9vZpQxbK7ReGp0NDgAU17JRoHtFsU_"
DATA_DRIVE   = "/content/drive/MyDrive/oceanembed/complete_model_ready_dataset.nc"  # fallback if download fails
SAVE_DRIVE   = "/content/drive/MyDrive/oceanembed/pinn_pipeline"                    # results backup

DATA = "/content/cube.nc"
if not os.path.exists(DATA):
    !gdown -q --fuzzy "https://drive.google.com/file/d/{DATA_FILE_ID}/view" -O {DATA}
if not os.path.exists(DATA) or os.path.getsize(DATA) < 1e6:
    print("gdown failed (file not shared as 'Anyone with the link'?) -> using DATA_DRIVE path")
    if DRIVE_OK: shutil.copy(DATA_DRIVE, DATA)
    else: raise RuntimeError("Download failed and Drive not mounted: share the file as 'Anyone with the link' and re-run")

RUNS, RESULTS, OUTPUTS = "/content/runs", "/content/results", "/content/outputs"
for d in (RUNS, RESULTS, OUTPUTS): os.makedirs(d, exist_ok=True)

def backup():
    if not DRIVE_OK:
        print("(no Drive - skipping backup)"); return
    os.makedirs(SAVE_DRIVE, exist_ok=True)
    for d in (RUNS, RESULTS, OUTPUTS):
        shutil.copytree(d, os.path.join(SAVE_DRIVE, os.path.basename(d)), dirs_exist_ok=True)
    os.sync()
    print("backed up to", SAVE_DRIVE)

import xarray as xr
_ds = xr.open_dataset(DATA)
print("data:", round(os.path.getsize(DATA)/1e6, 1), "MB |", dict(_ds.sizes), "|", list(_ds.data_vars))
_ds.close()


## 3. Data sanity check

In [ ]:
from oceanembed.data import DataConfig, OceanData
data = OceanData(DataConfig(path=DATA))
print("input channels:", len(data.channel_names), "(8 satellite fields x 3 days + 4 static)")
print("grid        :", data.H, "x", data.W, " depths:", len(data.depths), f"({data.depths[0]:.1f}-{data.depths[-1]:.0f} m)")
print("salinity    :", data.has_salinity)
print("ocean cells :", int(data.mask2d.sum()), "surface,", int(data.mask3d.sum()), "3D")
print("split (days):", {k: len(v) for k, v in data.idx.items()})
del data

## 4. Train: 4 models x 3 seeds (~15-20 min on a T4)
| run | encoder | physics |
|---|---|---|
| `unet` | CNN only | none |
| `oceanembed_nophys` | ViT + FNO | none |
| `oceanembed_constraints` | ViT + FNO | v2 constraint losses (MLD homogeneity, stability, steric) |
| `oceanembed_cyclone` | ViT + FNO | **cyclone-aware losses (D26, TCHP, D20, MLD) + stability + steric** — main model |

All runs: previous-2-days inputs, uncertainty head, same seeds, same schedule. Settings live in `configs/*.yaml`.

In [ ]:
SEEDS = [0, 1, 2]
VARIANTS = [("unet",                   "configs/ablation_unet.yaml"),
            ("oceanembed_nophys",      "configs/ablation_nophys.yaml"),
            ("oceanembed_constraints", "configs/ablation_constraints.yaml"),
            ("oceanembed_cyclone",     "configs/main.yaml")]           # main model
for name, cfg in VARIANTS:
    for s in SEEDS:
        print(f"\n########## {name}  seed {s} ##########")
        !python -W ignore -m oceanembed train --config {cfg} --data {DATA} --out {RUNS}/{name}_s{s} --seed {s}
backup()

## 5. Accuracy + cyclone products on held-out days (vs GLORYS)

In [ ]:
import glob
RUN_LIST = " ".join(sorted(glob.glob(f"{RUNS}/*_s*")))
MAIN = " ".join(sorted(glob.glob(f"{RUNS}/oceanembed_cyclone_s*")))
!python -W ignore -m oceanembed evaluate --data {DATA} --runs {RUN_LIST} --out {RESULTS}

## 6. Independent validation against ARGO floats (profiles + cyclone products)

In [ ]:
!python -W ignore -m oceanembed argo --data {DATA} --runs {RUN_LIST} --out {RESULTS} --cache {RESULTS}/argo_cache.csv

## 7. Calibrated uncertainty (3-seed ensemble) + final standardized output (mean + std NetCDF)

In [ ]:
!python -W ignore -m oceanembed uncertainty --data {DATA} --runs {MAIN} --out {RESULTS} --outputs {OUTPUTS} --argo_cache {RESULTS}/argo_cache.csv
backup()

## 8. Case study: deep depression over the NE Bay of Bengal, 1 Aug 2023

In [ ]:
!python -W ignore -m oceanembed casestudy --data {DATA} --runs {MAIN} --out {RESULTS} --argo_cache {RESULTS}/argo_cache.csv

## 9. Figures

In [ ]:
from IPython.display import Image, display
from oceanembed.eval import figures as P
S0 = ["climatology", "unet_s0", "oceanembed_nophys_s0", "oceanembed_constraints_s0", "oceanembed_cyclone_s0"]
figs = [
    P.per_depth_rmse(f"{RESULTS}/glorys_test_per_depth.csv", f"{RESULTS}/fig_rmse_by_depth.png", select=S0),
    P.training_curves([f"{RUNS}/{n}_s0" for n, _ in VARIANTS], f"{RESULTS}/fig_training.png"),
    P.argo_profiles(f"{RESULTS}/argo_matches_oceanembed_cyclone_s0.csv", f"{RESULTS}/fig_argo_profiles.png"),
    P.surface_vs_subsurface(f"{OUTPUTS}/oceanembed_temperature_daily_0p25.nc", 80, f"{RESULTS}/fig_depth_slices.png"),
    f"{RESULTS}/fig_uncertainty_calibration.png",
    f"{RESULTS}/fig_casestudy_timeseries.png",
    f"{RESULTS}/fig_casestudy_tchp_change.png",
]
for f in figs: display(Image(f))
backup()

## 10. Summary to paste back

In [ ]:
import pandas as pd, json
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
def typ(s): return s.str.replace(r"_s\d+$", "", regex=True)
g = pd.read_csv(f"{RESULTS}/glorys_test_overall.csv"); g["type"] = typ(g.model)
print("=== GLORYS held-out test days: mean (std) over 3 seeds ===")
print(g.groupby("type")[["rmse", "bias", "deep_T_inv_pct"]].agg(["mean", "std"]).round(3).to_string())
print()
print(g.groupby("type")[["D26_m_rmse", "D20_m_rmse", "TCHP_kJcm2_rmse", "MLD_m_rmse"]].agg(["mean", "std"]).round(2).to_string())
a = pd.read_csv(f"{RESULTS}/argo_summary.csv"); a["type"] = typ(a.model)
print("\n=== ARGO profiles (independent) ===")
print(a.groupby(["subset", "type"])[["rmse", "bias"]].agg(["mean", "std"]).round(3).to_string())
ap = pd.read_csv(f"{RESULTS}/argo_products_summary.csv"); ap["type"] = typ(ap.model)
print("\n=== Cyclone products vs ARGO ===")
print(ap.groupby(["subset", "type"])[["profiles", "D26_m_rmse", "D20_m_rmse", "TCHP_kJcm2_rmse", "MLD_m_rmse"]].mean().round(2).to_string())
print("\n=== Uncertainty coverage ===")
print(pd.read_csv(f"{RESULTS}/uncertainty_coverage.csv").round(3).to_string(index=False))
print("\n=== Case study (model vs GLORYS, box-mean daily evolution) ===")
print(pd.read_csv(f"{RESULTS}/casestudy_stats.csv").round(3).to_string(index=False))

In [ ]:
# Download everything (results, figures, NetCDF outputs, checkpoints) as one zip
!cd /content && zip -qr oceanembed_results.zip results outputs runs
from google.colab import files
files.download("/content/oceanembed_results.zip")